# Visualisations prediction results
This file contains the visualisation code for crating the visualised predictions from the paper. 

In [ ]:
# 1) Imports and project-root setup
from pathlib import Path
from typing import Dict, List

import os
import sys
import importlib

import matplotlib.pyplot as plt
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve().parent
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("CWD:", Path.cwd())
print("PROJECT_ROOT:", PROJECT_ROOT)

from src.utils.configs.training_config import load_config, build_run_plan
import src.utils.evaluation.evaluation as ev
import src.utils.visualisations.visualise_results as vis_paper

importlib.reload(ev)
importlib.reload(vis_paper)

Use the original heatmap layout and export each equation figure separately as vector PDF.

In [ ]:
USE_BEST = True
MODEL_ORDER = ["fno", "cape_fno", "late_fusion"]

PLOT_INDEX = {
    "advection": (0, 2),
    "burgers": (2, 2),
    "reactiondiffusion": (0, 0),
    "reactiondiffusion2d": (1, 5),
}

SAVE_NAME = {
    "advection": "heatmap_advection_A1A2.pdf",
    "burgers": "heatmap_burgers_B1B2.pdf",
    "reactiondiffusion": "heatmap_reactiondiffusion_C1C2.pdf",
    "reactiondiffusion2d": "heatmap_reactiondiffusion2d_D1D2.pdf",
}

def _build_best_bundle(equation: str) -> Dict:
    config_path = f"configs/training/{equation}_benchmark.yaml"
    cfg = load_config(config_path)
    runs = build_run_plan(cfg)

    val_rows: List[Dict] = []
    for run_cfg in runs:
        val_rows.append(ev.evaluate_run_validation(run_cfg, use_best=USE_BEST))

    val_df = pd.DataFrame(val_rows)
    _, best_combo = ev.select_best_combinations(val_df)

    selected = val_df.copy()
    selected["combination"] = selected["run_name"].map(ev.combination_key)
    selected = selected.merge(
        best_combo[["model", "combination"]],
        on=["model", "combination"],
        how="inner",
    )

    run_lookup = {r["name"]: r for r in runs}
    selected_runs = [run_lookup[name] for name in selected["run_name"].tolist()]

    test_rows = []
    for run_cfg in selected_runs:
        test_rows.extend(ev.evaluate_run_test(run_cfg, domains=("id",), use_best=USE_BEST))

    test_df = pd.DataFrame(test_rows)
    best_idx = test_df.groupby("model")["test_rmse"].idxmin()
    best_seed_per_model = test_df.loc[best_idx].sort_values("model").reset_index(drop=True)

    best_runs_by_model = {
        row["model"]: run_lookup[row["run_name"]]
        for _, row in best_seed_per_model.iterrows()
    }

    plot_id, plot_od = PLOT_INDEX[equation]
    return ev.collect_best_run_predictions(
        best_runs_by_model=best_runs_by_model,
        plot_id_idx=plot_id,
        plot_od_idx=plot_od,
        batch_size_test=10,
        use_best=USE_BEST,
    )

def _render_and_save(equation: str, label_pair: tuple):
    bundle = _build_best_bundle(equation)
    fig = vis_paper.plot_old_style_equation_heatmap(
        bundle=bundle,
        equation=equation,
        panel_labels=label_pair,
        model_order=MODEL_ORDER,
        font_size=9,
        max_models=3,
    )

    output_dir = PROJECT_ROOT / "outputs/Figures"
    output_dir.mkdir(exist_ok=True)
    save_path = output_dir / SAVE_NAME[equation]
    fig.savefig(save_path, format="pdf", bbox_inches="tight", pad_inches=0.12)
    plt.show()
    print(f"Saved vector PDF to {save_path}")
    return fig, bundle, save_path

1D advection (A1, A2)

In [ ]:
fig_adv, bundle_adv, save_adv = _render_and_save(
    equation="advection",
    label_pair=("A1", "A2"),
)

1D Burgers (B1, B2)

In [ ]:
fig_burg, bundle_burg, save_burg = _render_and_save(
    equation="burgers",
    label_pair=("B1", "B2"),
)

1D reaction-diffusion (C1, C2)

In [ ]:
fig_rd1d, bundle_rd1d, save_rd1d = _render_and_save(
    equation="reactiondiffusion",
    label_pair=("C1", "C2"),
)

2D reaction-diffusion (D1, D2)

In [ ]:
fig_rd2d, bundle_rd2d, save_rd2d = _render_and_save(
    equation="reactiondiffusion2d",
    label_pair=("D1", "D2"),
)

We merge the four PDFs into one stacked vector PDF
This keeps vector quality by merging PDF pages directly and adds grey horizontal separator lines between sections.

In [ ]:
import sys
from pathlib import Path

try:
    from pypdf import PdfReader, PdfWriter, PageObject, Transformation
except ImportError:
    import subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "pypdf"])
    from pypdf import PdfReader, PdfWriter, PageObject, Transformation

output_dir = PROJECT_ROOT / "outputs/Figures"
pdf_paths = [
    output_dir / SAVE_NAME["advection"],
    output_dir / SAVE_NAME["burgers"],
    output_dir / SAVE_NAME["reactiondiffusion"],
    output_dir / SAVE_NAME["reactiondiffusion2d"],
]

missing = [str(p) for p in pdf_paths if not p.exists()]
if missing:
    raise FileNotFoundError(f"Run Cells 5, 7, 9, and 11 first. Missing files: {missing}")

pages = [PdfReader(str(p)).pages[0] for p in pdf_paths]
widths = [float(p.mediabox.width) for p in pages]
heights = [float(p.mediabox.height) for p in pages]

max_width = max(widths)
gap = 24.0
total_height = sum(heights) + gap * (len(pages) - 1)

# Create a vector separator line page with the same width as the merged page.
sep_h = 8.0
sep_fig = plt.figure(figsize=(max_width / 72.0, sep_h / 72.0), dpi=72)
sep_ax = sep_fig.add_axes([0, 0, 1, 1])
sep_ax.plot([0.02, 0.98], [0.5, 0.5], color="0.7", linewidth=1.0)
sep_ax.set_xlim(0, 1)
sep_ax.set_ylim(0, 1)
sep_ax.axis("off")
separator_pdf = output_dir / "_separator_line_tmp.pdf"
sep_fig.savefig(separator_pdf, format="pdf", bbox_inches=None, pad_inches=0)
plt.close(sep_fig)
separator_page = PdfReader(str(separator_pdf)).pages[0]

merged_page = PageObject.create_blank_page(width=max_width, height=total_height)
cursor_y = total_height

for idx, page in enumerate(pages):
    w = float(page.mediabox.width)
    h = float(page.mediabox.height)
    x = (max_width - w) / 2.0
    y = cursor_y - h

    merged_page.merge_transformed_page(
        page,
        Transformation().translate(tx=x, ty=y),
    )

    cursor_y = y
    if idx < len(pages) - 1:
        sep_y = cursor_y - (gap / 2.0) - (sep_h / 2.0)
        merged_page.merge_transformed_page(
            separator_page,
            Transformation().translate(tx=0, ty=sep_y),
        )
        cursor_y -= gap

writer = PdfWriter()
writer.add_page(merged_page)
merged_path = output_dir / "all_equations_stacked_merged.pdf"
with open(merged_path, "wb") as f:
    writer.write(f)

try:
    separator_pdf.unlink()
except OSError:
    pass

print(f"Saved merged vector PDF to {merged_path}")